# LangGraph route walkthrough

**Owner:** Ibraheem

Executed walkthrough of the four real routes (literature, trend, hybrid, fallback), the retry loop and the SQLite checkpointer.

In [1]:
# Data paths: run this cell first.
import sys
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
EMBEDDINGS_PATH = REPO_DIR / "data" / "embeddings.npy"
DATA_PATH, EMBEDDINGS_PATH

(PosixPath('/media/ibraheem/HDD/research_paper_discovery/data/papers.parquet'),
 PosixPath('/media/ibraheem/HDD/research_paper_discovery/data/embeddings.npy'))

## Build the graph

Call src.graph.build_graph with a checkpointer.

In [2]:
from src.graph import build_graph, get_checkpointer
from src.settings_p2 import (
    CHECKPOINT_DB_PATH,
    DOC_TOPICS_PATH,
    EMBEDDINGS_PATH,
    MAX_RETRIES,
    TOPIC_SHARE_DUMMY_PATH,
    TOPIC_SHARE_PATH,
)

checkpointer = get_checkpointer()
graph = build_graph(checkpointer)
print("compiled graph:", type(graph).__name__)
print("checkpointer db:", CHECKPOINT_DB_PATH, "exists:", CHECKPOINT_DB_PATH.exists())
print("MAX_RETRIES:", MAX_RETRIES)
print("real topic share:", TOPIC_SHARE_PATH, "exists:", TOPIC_SHARE_PATH.exists())
print("dummy topic share exists:", TOPIC_SHARE_DUMMY_PATH.exists())
print("embeddings:", EMBEDDINGS_PATH, "exists:", EMBEDDINGS_PATH.exists())
print("doc topics:", DOC_TOPICS_PATH, "exists:", DOC_TOPICS_PATH.exists())

compiled graph: CompiledStateGraph
checkpointer db: /media/ibraheem/HDD/research_paper_discovery/cache/phase2_checkpoints.sqlite exists: True
MAX_RETRIES: 2
real topic share: /media/ibraheem/HDD/research_paper_discovery/results/topic_share.csv exists: True
dummy topic share exists: True
embeddings: /media/ibraheem/HDD/research_paper_discovery/data/embeddings.npy exists: True
doc topics: /media/ibraheem/HDD/research_paper_discovery/data/doc_topics.parquet exists: True


## Define six demo questions

One fallback, one literature, one trend, one hybrid,
one literature with 'retry-test', and one hybrid with 'retry-test'.

In [3]:
QUESTIONS = [
    ("fallback", "What is the weather tomorrow?"),
    ("literature", "Find papers about graph neural networks published after 2021 in cs.LG"),
    ("trend", "How has topic share changed over time?"),
    ("hybrid", "Show papers about CLIP and how their topic share trends over time"),
    ("literature + retry", "retry-test: find papers about GNNs"),
    ("hybrid + retry", "retry-test: show papers about CLIP and how their topic share trends over time"),
]
for label, question in QUESTIONS:
    print(f"{label}: {question}")

fallback: What is the weather tomorrow?
literature: Find papers about graph neural networks published after 2021 in cs.LG
trend: How has topic share changed over time?
hybrid: Show papers about CLIP and how their topic share trends over time
literature + retry: retry-test: find papers about GNNs
hybrid + retry: retry-test: show papers about CLIP and how their topic share trends over time


## Run each question

Call src.graph.run_question and record the final GraphState.

In [4]:
from src.graph import run_question

states = []
for i, (label, question) in enumerate(QUESTIONS):
    state = run_question(question, thread_id=f"demo-{i}")
    states.append((label, question, state))
    print("=" * 70)
    print(f"case={label}")
    print(f"question={question}")
    print(
        f"route={state.get('route')} "
        f"retry_count={state.get('retry_count')} "
        f"llm_calls={state.get('llm_calls')}"
    )
    print(f"answer={state.get('answer')}")
    print(f"citations={state.get('citations')}")
    print(f"low_confidence={state.get('low_confidence')}")

router_node LLM failed (Missing LLM API key: set GEMINI_API_KEY or GROQ_API_KEY in .env); using keyword fallback


case=fallback
question=What is the weather tomorrow?
route=fallback retry_count=0 llm_calls=0
answer=I focus on arXiv papers in cs.LG, cs.CL and cs.CV (2018-2024) and their trends over time. Try asking for papers on a topic, how a topic has changed over time, or both together.
citations=[]
low_confidence=True


router_node LLM failed (Missing LLM API key: set GEMINI_API_KEY or GROQ_API_KEY in .env); using keyword fallback


Query rewrite unavailable: Missing LLM API key: set GEMINI_API_KEY or GROQ_API_KEY in .env


Retrieval resources loaded: FAISS ntotal=119919, embeddings=(119919, 384), doc_topics rows=119919, papers=119919


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

... (21 outputs left out)

Query rewrite unavailable: Missing LLM API key: set GEMINI_API_KEY or GROQ_API_KEY in .env


Query rewrite unavailable: Missing LLM API key: set GEMINI_API_KEY or GROQ_API_KEY in .env


synthesize_node LLM failed (Missing LLM API key: set GEMINI_API_KEY or GROQ_API_KEY in .env)


case=hybrid + retry
question=retry-test: show papers about CLIP and how their topic share trends over time
route=hybrid retry_count=2 llm_calls=3
answer=Found 5 related papers: Forgotten Knowledge: Examining the Citational Amnesia in NLP, Dynamic and Static Topic Model for Analyzing Time-Series Document Collections, Changepoint Analysis of Topic Proportions in Temporal Text Data. Series / Time / Forecasting changed from 0.0142 (2018Q1) to 0.0268 (2024Q4) (slope +0.00051/quarter).
citations=[{'arxiv_id': '2305.18554', 'title': 'Forgotten Knowledge: Examining the Citational Amnesia in NLP'}, {'arxiv_id': '1805.02203', 'title': 'Dynamic and Static Topic Model for Analyzing Time-Series Document Collections'}, {'arxiv_id': '2112.00827', 'title': 'Changepoint Analysis of Topic Proportions in Temporal Text Data'}, {'arxiv_id': '1806.01185', 'title': 'History Playground: A Tool for Discovering Temporal Trends in Massive Textual Corpora'}, {'arxiv_id': '2310.03928', 'title': 'Exploring the evol

## Inspect routes and LLM calls

Check the route matches and count llm_calls per path.

In [5]:
from IPython.display import display
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "case": label,
            "route": state.get("route"),
            "retry_count": state.get("retry_count"),
            "llm_calls": state.get("llm_calls"),
            "n_citations": len(state.get("citations", [])),
            "trend_topic": (state.get("trend_data") or {}).get("topic_id"),
        }
        for label, _, state in states
    ]
)
display(summary)

,case,route,retry_count,llm_calls,n_citations,trend_topic
0,fallback,fallback,0,0,0,NaN
1,literature,literature,2,3,0,NaN
2,trend,trend,0,1,0,8.0
3,hybrid,hybrid,2,3,5,8.0
4,literature + retry,literature,2,3,0,NaN
5,hybrid + retry,hybrid,2,3,5,8.0


## Inspect the retry loop

Confirm retry_count reaches MAX_RETRIES and low_confidence is True.

In [6]:
retry_cases = [state for label, _, state in states if "retry" in label]
assert retry_cases, "no retry case ran"
for state in retry_cases:
    print(
        "retry_count:", state.get("retry_count"),
        "MAX_RETRIES:", MAX_RETRIES,
        "low_confidence:", state.get("low_confidence"),
    )
    assert state.get("retry_count") == MAX_RETRIES
    assert state.get("low_confidence") is True
print("retry loop verified: grade 'no' rewrote until MAX_RETRIES, then flagged low confidence")

retry_count: 2 MAX_RETRIES: 2 low_confidence: True
retry_count: 2 MAX_RETRIES: 2 low_confidence: True
retry loop verified: grade 'no' rewrote until MAX_RETRIES, then flagged low confidence


## Results

The executed cells above print the route, retry count, llm_calls, answer and citations for all six questions, then summarise them in a table and verify the retry loop. The graph topology is shown below.

## Graph topology

```mermaid
flowchart TD
    START([start]) --> router
    router -->|literature| rewrite
    router -->|hybrid| rewrite
    router -->|trend| trend_lookup
    router -->|fallback| fallback
    rewrite --> retrieve
    retrieve --> grade
    grade -->|"grade=no and retry_count < MAX_RETRIES"| rewrite
    grade -->|"hybrid otherwise"| trend_lookup
    grade -->|"otherwise"| answer
    trend_lookup --> synthesize
    answer --> END([end])
    synthesize --> END
    fallback --> END
```

In [7]:
MERMAID = """flowchart TD
    START([start]) --> router
    router -->|literature| rewrite
    router -->|hybrid| rewrite
    router -->|trend| trend_lookup
    router -->|fallback| fallback
    rewrite --> retrieve
    retrieve --> grade
    grade -->|"grade=no and retry_count < MAX_RETRIES"| rewrite
    grade -->|"hybrid otherwise"| trend_lookup
    grade -->|"otherwise"| answer
    trend_lookup --> synthesize
    answer --> END([end])
    synthesize --> END
    fallback --> END"""
print(MERMAID)

flowchart TD
    START([start]) --> router
    router -->|literature| rewrite
    router -->|hybrid| rewrite
    router -->|trend| trend_lookup
    router -->|fallback| fallback
    rewrite --> retrieve
    retrieve --> grade
    grade -->|"grade=no and retry_count < MAX_RETRIES"| rewrite
    grade -->|"hybrid otherwise"| trend_lookup
    grade -->|"otherwise"| answer
    trend_lookup --> synthesize
    answer --> END([end])
    synthesize --> END
    fallback --> END
